# Fine-tuning PhoBERT with LoRA

PhoBERT-base-v2 from Hugging Face, fine-tuned on the reviews by masked language modelling: the reviews have no labels, so it learns their language by guessing words hidden from it. With LoRA, PhoBERT's own weights stay as they are and only small added matrices in its attention are trained.

In [11]:
import unicodedata

import pandas as pd


def clean(text: str) -> str:
    """Lowercased, with only letters, numbers, punctuation and spaces: emoji and other symbols are dropped.
    NFC first, so each Vietnamese letter is one character, not a letter plus combining accent marks."""
    text = unicodedata.normalize("NFC", text)
    text = "".join(ch for ch in text if unicodedata.category(ch)[0] in "LNPZ")
    return " ".join(text.split()).lower()


data = pd.read_csv("data/bert_food_history.csv", usecols=["content"])
data["content"] = data["content"].fillna("").map(clean)
# Only written reviews: purchases not rated, or rated with just the stars, are dropped
data = data[data["content"] != ""].reset_index(drop=True)

print(f"{len(data)} reviews")
data.head()

6744 reviews


,content
0,"ăn được đó, ít quá, không đã, giá hợp lý, soco..."
1,"ngon hết nấc, fan cà phê như anh thì khỏi chê,..."
2,"yêu ghê đáng đồng tiền, vị đậm đà, em vốn thíc..."
3,"ok thôi, đắt so với chất lượng, anh mê hải sản..."
4,"good, nêm hơi mặn. cảm ơn em nha"


In [12]:
data.shape

(6744, 1)

In [13]:
from sklearn.model_selection import train_test_split

# 90% to train on, 10% held back to check it on reviews it hasn't seen
train, val = train_test_split(data, test_size=0.1, random_state=0)
print("training:", train.shape, "validation:", val.shape)

training: (6069, 1) validation: (675, 1)


In [14]:
import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer

PHOBERT = "vinai/phobert-base-v2"
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

# PhoBERT reads word-segmented text ("Trà_sữa trân_châu"): segment with underthesea's word_tokenize first.
# Inputs up to 256 tokens.
tokenizer = AutoTokenizer.from_pretrained(PHOBERT)
# On MPS (Apple GPUs) the default attention can't do dropout, which training needs: use the plain one there
attention = "eager" if device == "mps" else "sdpa"
# With its masked-word head: what it's fine-tuned on
model = AutoModelForMaskedLM.from_pretrained(PHOBERT, attn_implementation=attention).to(device)
model.train()  # dropout on, every weight trainable

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{PHOBERT} on {device}: {trainable / 1e6:.0f}M trainable parameters")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

vinai/phobert-base-v2 on mps: 135M trainable parameters


In [15]:
from peft import LoraConfig, PeftModel, get_peft_model

# LoRA: PhoBERT's weights frozen, and next to each attention query and value layer a pair of small
# matrices (rank 8) that learn the change instead: well under 1% of the parameters to train
LORA = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.1, target_modules=["query", "value"])
if not isinstance(model, PeftModel):  # run twice, it would add LoRA on top of LoRA
    model = get_peft_model(model, LORA)
model.train()
model.print_trainable_parameters()

trainable params: 294,912 || all params: 135,358,721 || trainable%: 0.2179


In [16]:
from underthesea import word_tokenize

texts = ["Trà sữa trân châu đường đen", "Matcha latte ít ngọt, ngon xỉu"]
batch = tokenizer([word_tokenize(t, format="text") for t in texts], padding=True, truncation=True, max_length=256, return_tensors="pt").to(device)
print({name: tuple(tensor.shape) for name, tensor in batch.items()})  # input: (batch, seq_len)
with torch.no_grad():
    print("last_hidden_state:", tuple(model.roberta(**batch).last_hidden_state.shape))  # model.roberta: PhoBERT without the head  # output: (batch, seq_len, 768)

{'input_ids': (2, 11), 'attention_mask': (2, 11)}
last_hidden_state: (2, 11, 768)


In [17]:
from torch.utils.data import DataLoader
from transformers import DataCollatorForLanguageModeling


def encode(texts):
    """Word-segmented as PhoBERT expects and cut to its 256 tokens, not padded yet"""
    return [tokenizer(word_tokenize(t, format="text"), truncation=True, max_length=256) for t in texts]


class FrontMaskingCollator(DataCollatorForLanguageModeling):
    """Pads each batch to its longest review and hides tokens for the model to guess (labels: the hidden ones),
    the earlier in the review the likelier: `start` for the first token, falling evenly to `end` for the last.
    15% on average, as BERT's usual masking."""

    def __init__(self, tokenizer, start=0.25, end=0.05):
        super().__init__(tokenizer, mlm_probability=(start + end) / 2)
        self.start, self.end = start, end

    def torch_mask_tokens(self, inputs, special_tokens_mask=None, offset_mapping=None):
        labels = inputs.clone()
        if special_tokens_mask is None:
            special_tokens_mask = torch.tensor(
                [self.tokenizer.get_special_tokens_mask(row, already_has_special_tokens=True) for row in labels.tolist()]
            )
        words = ~special_tokens_mask.bool()  # <s>, </s> and padding are never hidden
        position = (words.cumsum(1) - 1).clamp(min=0) / (words.sum(1, keepdim=True) - 1).clamp(min=1)  # 0 first, 1 last
        probability = (self.start + (self.end - self.start) * position) * words
        masked = torch.bernoulli(probability, generator=self.generator).bool()
        labels[~masked] = -100  # the loss is only on the hidden tokens

        # As BERT does with them: 80% become <mask>, 10% a random token, 10% stay as they are
        replaced = torch.bernoulli(torch.full(labels.shape, self.mask_replace_prob), generator=self.generator).bool() & masked
        inputs[replaced] = self.tokenizer.mask_token_id
        random_share = self.random_replace_prob / (1 - self.mask_replace_prob)
        randomised = torch.bernoulli(torch.full(labels.shape, random_share), generator=self.generator).bool() & masked & ~replaced
        inputs[randomised] = torch.randint(len(self.tokenizer), labels.shape, generator=self.generator)[randomised]
        return inputs, labels


collator = FrontMaskingCollator(tokenizer)
train_loader = DataLoader(encode(train["content"]), batch_size=16, shuffle=True, collate_fn=collator)

# The validation reviews are masked once, so every check scores the same hidden words
torch.manual_seed(0)
val_batches = list(DataLoader(encode(val["content"]), batch_size=32, collate_fn=collator))

batch = next(iter(train_loader))
print(f"{len(train_loader)} training batches, {len(val_batches)} validation batches")
print({name: tuple(tensor.shape) for name, tensor in batch.items()})  # each (batch, seq_len)

380 training batches, 22 validation batches
{'input_ids': (16, 25), 'attention_mask': (16, 25), 'labels': (16, 25)}


In [ ]:
import math

from tqdm.auto import tqdm
from transformers import get_linear_schedule_with_warmup

EPOCHS = 10
EVAL_EVERY = 50  # training steps between validation checks, for the graph
# Only the LoRA matrices train, and at a higher learning rate than full fine-tuning (2e-5) would use
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=3e-4, weight_decay=0.01)
steps = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=steps // 10, num_training_steps=steps)


@torch.no_grad()
def validation_loss():
    model.eval()
    losses = [model(**{k: v.to(device) for k, v in b.items()}).loss.item() for b in val_batches]
    model.train()
    return sum(losses) / len(losses)


# For the graph: the loss of every training step, and (step, loss) of every validation check
history = {"train": [], "val": [(0, validation_loss())]}
print(f"before: validation loss {history['val'][0][1]:.3f}")
step = 0
for epoch in range(1, EPOCHS + 1):
    total = 0.0
    for batch in tqdm(train_loader, desc=f"epoch {epoch}"):
        loss = model(**{k: v.to(device) for k, v in batch.items()}).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        step += 1
        total += loss.item()
        history["train"].append(loss.item())
        if step % EVAL_EVERY == 0:
            history["val"].append((step, validation_loss()))
    if history["val"][-1][0] != step:  # and at the end of every epoch
        history["val"].append((step, validation_loss()))
    val_loss = history["val"][-1][1]
    print(f"epoch {epoch}: training loss {total / len(train_loader):.3f}, validation loss {val_loss:.3f} (perplexity {math.exp(val_loss):.1f})")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

SURFACE, INK, INK_SECONDARY, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e6e5e1"
TRAINING, VALIDATION = "#2a78d6", "#eb6834"
SMOOTH = 25  # batches averaged for the training line: one batch's loss alone is noisy

losses = np.array(history["train"])
smooth = min(SMOOTH, len(losses))
training = np.convolve(losses, np.ones(smooth) / smooth, mode="valid")
training_steps = np.arange(smooth, len(losses) + 1)
val_steps, val_losses = map(np.array, zip(*history["val"]))
per_epoch = len(train_loader)

fig, ax = plt.subplots(figsize=(9, 4.8), facecolor=SURFACE, layout="constrained")
ax.set_facecolor(SURFACE)
ax.plot(training_steps, training, color=TRAINING, linewidth=2, label=f"Training (average of {smooth} batches)")
ax.plot(val_steps, val_losses, color=VALIDATION, linewidth=2, marker="o", markersize=6,
        markeredgecolor=SURFACE, markeredgewidth=2, label="Validation")

# Where each epoch starts
for e in range(1, int(np.ceil(len(losses) / per_epoch))):
    ax.axvline(e * per_epoch, color=GRID, linewidth=1, zorder=0)
    ax.text(e * per_epoch, 1, f" epoch {e + 1}", transform=ax.get_xaxis_transform(), color=MUTED, fontsize=9, va="top")

# The latest value at the end of each line
for x, y in [(training_steps[-1], training[-1]), (val_steps[-1], val_losses[-1])]:
    ax.annotate(f"{y:.2f}", (x, y), xytext=(6, 0), textcoords="offset points", va="center", color=INK_SECONDARY, fontsize=10)

start, end = val_losses[0], val_losses[-1]
ax.set_title("Masked-word loss while fine-tuning", loc="left", color=INK, fontsize=13, pad=22)
ax.text(0, 1.02, f"Validation loss {start:.2f} → {end:.2f}, perplexity {np.exp(start):.0f} → {np.exp(end):.0f}",
        transform=ax.transAxes, color=INK_SECONDARY, fontsize=10)
ax.set_xlabel("Training step", color=INK_SECONDARY)
ax.set_ylabel("Loss (lower is better)", color=INK_SECONDARY)
ax.set_xlim(0, len(losses) * 1.04)
ax.grid(axis="y", color=GRID, linewidth=1)
ax.set_axisbelow(True)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
for side in ["left", "bottom"]:
    ax.spines[side].set_color(MUTED)
ax.tick_params(colors=INK_SECONDARY)
ax.legend(frameon=False, labelcolor=INK_SECONDARY, loc="upper right")
plt.show()

In [ ]:
SAVE_TO = "models/phobert-lora"
model.save_pretrained(SAVE_TO)
tokenizer.save_pretrained(SAVE_TO)

('models/phobert-lora/tokenizer_config.json',
 'models/phobert-lora/vocab.txt',
 'models/phobert-lora/bpe.codes',
 'models/phobert-lora/added_tokens.json')